# Day 05 下午学生项目：电商用户多维分析

**小组编号：** 第 1 组
**成员：** 张三
**专题方向：** A（TenureGroup 用户生命周期）

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "G01"
MEMBERS = ["张三"]
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)

### 检查点0

- [ ] 已填写组号、成员和专题；
- [ ] Notebook文件名包含组号；
- [ ] 输出目录中的组号正确。

## 任务1：加载并验收数据（必做）

In [ ]:
# TODO 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)

# 修复 TenureGroup：Tenure=0 的用户应归入 "0-12 months"
df["TenureGroup"] = df["TenureGroup"].fillna("0-12 months")

# TODO 2：输出shape、前5行和字段类型
print("数据形状：", df.shape)
print("\n前5行：")
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("类型"))

# TODO 3：计算以下验收结果
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": df["CustomerID"].duplicated().sum(),
    "核心字段缺失数": df.isna().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}
validation

In [ ]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")

**数据粒度：** 请用一句话填写：
每一行代表一位电商用户，包含其人口属性、使用行为、消费记录及流失标记共22个字段。

## 任务2：公共基础指标（必做）

In [ ]:
# TODO：构建overall_metrics DataFrame，至少包含以下指标：
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数

overall_metrics = pd.DataFrame({
    "指标": [
        "用户数", "流失人数", "流失率(%)", "平均订单数", "订单数中位数",
        "平均优惠券数", "平均返现", "平均App时长(小时)", "平均满意度",
        "平均距上次下单天数", "平均使用时长(月)", "移动端登录占比(%)",
        "投诉率(%)", "平均注册设备数",
    ],
    "数值": [
        len(df),
        df["Churn"].sum(),
        df["Churn"].mean() * 100,
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
        df["Tenure"].mean(),
        df["IsMobileLogin"].mean() * 100,
        df["Complain"].mean() * 100,
        df["NumberOfDeviceRegistered"].mean(),
    ]
})

display(overall_metrics)

In [ ]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")

## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [ ]:
# TODO：填写你的分组字段 — 专题A：TenureGroup 用户生命周期
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", lambda x: x.mean() * 100),
    平均订单数=("OrderCount", "mean"),
    订单数中位数=("OrderCount", "median"),
    平均返现=("CashbackAmount", "mean"),
    平均App时长=("HourSpendOnApp", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
    平均使用时长=("Tenure", "mean"),
).reset_index()

# 按流失率降序排列
segment_analysis = segment_analysis.sort_values("流失率", ascending=False).reset_index(drop=True)

# TODO：重置索引、排序并展示
display(segment_analysis)

In [ ]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

### 专题分析记录

**数据现象：**
"0-12 months"组用户数最多（3734人，含508名Tenure=0的用户），流失率最高（22.84%），平均订单数最低（2.60）；"13-24 months"组流失率大幅下降至6.48%，平均订单数升至3.70；"25-36 months"和"37+ months"组流失率均为0%，但"37+ months"组仅4人，样本过小不具备统计意义。

**可能解释：**
新用户（0-12个月）流失率高可能与初次体验不佳或未形成使用习惯有关，这属于用户生命周期的早期筛选阶段；随着使用时长增加，留存用户可能是对平台更满意的高粘性群体（幸存者偏差），因此流失率随Tenure增加而递减。需要注意的是，相关关系不等于因果关系——使用时间长并不直接"导致"低流失，可能是高满意度用户本身就同时具备长使用时长和低流失倾向。

## 任务4：双维度交叉分析（必做）

In [ ]:
# TODO：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "CityTier"

# TODO：按两个维度统计用户数、流失人数、流失率，以及至少一个行为指标
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", lambda x: x.mean() * 100),
    平均订单数=("OrderCount", "mean"),
    平均返现=("CashbackAmount", "mean"),
).reset_index()

# TODO：新增"样本提示"列；用户数<30标记为"小样本"，否则为"可观察"
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(
    lambda x: "小样本" if x < 30 else "可观察"
)

# TODO：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False).reset_index(drop=True)
display(cross_analysis)

In [ ]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

### 双维分析记录

**最值得关注的组合：**
"0-12 months" × "CityTier 2（二线城市）"，以及"0-12 months" × "CityTier 3（三线城市）"。

**该组合的样本量与流失率：**
- 0-12 months / CityTier 2：152人，流失率最高达31.58%；
- 0-12 months / CityTier 3：1205人，流失率次高为27.05%；
- 0-12 months / CityTier 1：2377人，流失率相对较低为20.15%。

**为什么不能直接下因果结论：**
首先，这是观察性数据而非随机对照实验，流失率的差异可能受混淆变量影响（如不同城市层级的用户群体本身在收入、年龄等方面就存在结构性差异）；其次，CityTier 2 在 0-12 months 组仅152人，而 CityTier 1 有 2377 人，样本量不均可能放大极端值的影响。只能说"0-12个月且位于二三线城市的用户流失率更高"这一相关性值得进一步验证，但不能断言城市层级导致了流失。

## 任务5：报表输出与回读验证（必做）

In [ ]:
# TODO：将三个表导出到OUTPUT_DIR
# 文件名必须为：overall_metrics.csv、segment_analysis.csv、cross_analysis.csv
# 要求：index=False，encoding="utf-8-sig"

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# TODO：循环导出并重新读取；打印每个文件的shape
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    print(f"{filename}: 导出shape={table.shape}, 回读shape={reloaded.shape}")

In [ ]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

## 任务6：结论、限制与建议（必做）

### 结论1

在 **TenureGroup 为 "0-12 months"** 用户中，**流失率** 为 **22.84%**，与 **"13-24 months" 组（6.48%）** 相比 **高出约 16.4 个百分点，是后者的 3.5 倍**。对应证据表：**segment_analysis.csv**。

### 结论2

在 "0-12 months" 新用户中，三线城市（CityTier 3）和二线城市（CityTier 2）的流失率分别为 27.05% 和 31.58%，均高于一线城市（20.15%），这可能与低线城市用户的替代平台选择更多或平台服务覆盖不足有关，需结合物流、客服等数据进一步验证。对应证据表：**cross_analysis.csv**。

### 结论3

用户整体投诉率为 28.49%，平均满意度仅 3.07/5，且平均订单数仅 2.96 单，说明平台在用户体验和复购激励方面仍有较大的提升空间。约 71% 的用户通过移动端登录，移动端体验优化应成为运营重点。

### 分析限制

本数据集缺少订单时间戳和订单金额字段，因此不能计算 GMV、客单价、时间趋势（如月度流失变化）或用户消费周期。此外，数据为截面快照，无法区分"已流失"和"从未活跃"的用户，也无法判断流失发生的时间点。TenureGroup 中 "37+ months" 仅 4 人，任何针对该组的结论均不可靠。返现（CashbackAmount）不等同于消费金额，不可用于收入分析。

### 运营建议与验证方式

**建议：** 针对 0-12 个月新用户（尤其二三线城市），设计"新手任务+首单返现"的留存激励组合，在注册后 7 天内引导完成首次下单。

**验证方式：** 需要 A/B 实验数据——随机分配新用户到实验组（有新手激励）和对照组（无干预），对比两组 30 天/90 天留存率和下单转化率。同时需要用户注册日期和订单日期，以计算精确的留存曲线。

## 拓展任务（选做）

In [ ]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# TODO（选做）—— 订单活跃度分层
df["OrderActivity"] = pd.qcut(
    df["OrderCount"], q=4, labels=["低活跃", "中活跃", "高活跃", "超高活跃"]
)
print("===== 订单活跃度分层分布 =====")
print(df["OrderActivity"].value_counts().sort_index())
print()

# 活跃度 × 流失率
activity_churn = df.groupby("OrderActivity", observed=False).agg(
    用户数=("CustomerID", "count"),
    流失率=("Churn", lambda x: x.mean() * 100),
).reset_index()
print("===== 订单活跃度 × 流失率 =====")
display(activity_churn)

## 提交前检查

- [ ] 已填写组号、成员和专题；
- [ ] 已重启内核并从头运行成功；
- [ ] 所有比例表都包含样本量；
- [ ] 三个CSV已导出并回读；
- [ ] 至少3条结论可对应到具体表格；
- [ ] 已写明分析限制和验证建议；
- [ ] 没有把返现写成消费额，没有把相关写成因果。